<a href="https://colab.research.google.com/github/Smyles019/html-login-form-detector/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import pickle
from collections import Counter
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

# 1. Load and merge data files
file_path_1 = 'train_login_form.csv'
file_path_2 = 'train_no_form.csv'

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)
df = pd.concat([df1, df2], ignore_index=True)
print(f"Total records loaded: {len(df)} (file1: {len(df1)}, file2: {len(df2)})")

# 1-1. Handle missing values
n_missing = df['html_signature'].isna().sum()
if n_missing:
    print(f"⚠️ Found {n_missing} missing html_signature values → replacing with empty string")
df['html_signature'] = df['html_signature'].fillna('')

# 1-2. Remove duplicate sha256 (in case the same page appears in both files)
before = len(df)
df = df.drop_duplicates(subset='sha256', keep='first').reset_index(drop=True)
if before != len(df):
    print(f"⚠️ Removed {before - len(df)} duplicate sha256 records")

# 1-3. Check class distribution
print("\nLabel distribution:")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True).round(3))

# 2. Extract structural numeric features
def extract_structural_features(sig: str) -> dict:
    max_depth = 0
    curr_depth = 0
    for char in sig:
        if char == '(':
            curr_depth += 1
            max_depth = max(max_depth, curr_depth)
        elif char == ')':
            curr_depth = max(0, curr_depth - 1)  # guard against unbalanced parentheses

    tags = re.findall(r'[a-zA-Z0-9]+', sig)
    tag_counts = Counter(tags)
    form_count = tag_counts.get("form", 0)
    input_count = tag_counts.get("input", 0)

    return {
        "signature_length": len(sig),
        "max_depth": max_depth,
        "total_tags": len(tags),
        "unique_tags_count": len(tag_counts),
        "form_count": form_count,
        "input_count": input_count,
        "button_count": tag_counts.get("button", 0),
        "div_count": tag_counts.get("div", 0),
        "script_count": tag_counts.get("script", 0),
        "a_count": tag_counts.get("a", 0),
        "img_count": tag_counts.get("img", 0),
        "input_per_form_ratio": input_count / form_count if form_count > 0 else 0.0,
    }

df_struct = pd.DataFrame([extract_structural_features(sig) for sig in df['html_signature']])

# 3. Extract N-gram features (keep sparse)
cleaned_signatures = [re.sub(r'[\(\)]+', ' ', sig).strip() for sig in df['html_signature']]
vectorizer = CountVectorizer(ngram_range=(2, 3), token_pattern=r'\b\w+\b', min_df=2)
ngram_matrix = vectorizer.fit_transform(cleaned_signatures)  # keep as sparse matrix

# Save the vectorizer so it can be reused (transform) on test data later
with open('ngram_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

ngram_cols = [f"ngram_{name}" for name in vectorizer.get_feature_names_out()]

# 4. Final combination — keep structural features dense, n-gram sparse, merge later at model input time
meta_cols = df[['url', 'sha256', 'label']].reset_index(drop=True)
struct_df = df_struct.reset_index(drop=True)

X_preprocessed = pd.concat([meta_cols, struct_df], axis=1)
X_ngram_sparse = ngram_matrix  # scipy.sparse.csr_matrix, combine with sparse.hstack when training

print(f"\nStructural feature shape: {struct_df.shape}")
print(f"N-gram feature shape (sparse): {ngram_matrix.shape}")

# =========================================================
# 📊 [Output] Key feature statistics and top N-gram patterns
# =========================================================
print("\n" + "="*50)
print("1. Overall Statistics for Structural Numeric Features")
print("="*50)
struct_cols = ['max_depth', 'total_tags', 'form_count', 'input_count', 'button_count', 'script_count']
print(df_struct[struct_cols].describe().round(2))

if 'label' in df.columns and df['label'].nunique() > 1:
    print("\n" + "="*50)
    print("2. Class-wise Comparison of Structural Feature Means")
    print("="*50)
    comparison_df = pd.concat([df['label'], df_struct[struct_cols]], axis=1)
    print(comparison_df.groupby('label').mean().round(2))

print("\n" + "="*50)
print("3. Top 10 Most Frequent N-gram Patterns")
print("="*50)
ngram_sums = pd.Series(ngram_matrix.sum(axis=0).A1, index=ngram_cols)
top_ngrams = ngram_sums.sort_values(ascending=False).head(10)
for idx, (ngram_name, count) in enumerate(top_ngrams.items(), 1):
    clean_name = ngram_name.replace('ngram_', '')
    print(f"{idx:2d}. Tag pattern [{clean_name}]: appeared {int(count)} times")